# ForgeSight — Fine-tune Qwen2.5-3B on a free Colab T4

This notebook runs the QLoRA fine-tune end-to-end on a **free Colab T4 (16 GB)** and downloads the
GGUF you then serve locally via Ollama.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

> ⚠️ **Colab runs on a remote Google VM — it cannot read paths on your laptop** (`C:\...`).
> You must *upload* the training files. Cell 2 opens a file picker: choose either a **zip of your
> local `finetune/` folder**, or the 3 loose files `colab_train.py`, `sft_train.jsonl`,
> `sft_eval.jsonl`. Because of that prompt, run the cells **top to bottom** (Run all also works —
> it just pauses at the upload dialog).

Pipeline: upload files → install Unsloth → train (`finetune/colab_train.py`) → export `Q4_K_M.gguf`
→ download. Then locally: `ollama create qwen-forgesight -f finetune/Modelfile`.

In [6]:
#@title 1. Confirm the T4 GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [ ]:
#@title 2. Upload the training files from your laptop
# Colab runs on a remote Google VM, so it cannot read paths like C:\... on your machine.
# Upload the files instead. Pick EITHER:
#   (a) a single .zip of your local `finetune/` folder, OR
#   (b) the 3 loose files: colab_train.py, sft_train.jsonl, sft_eval.jsonl
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

os.makedirs("finetune/dataset", exist_ok=True)
print("Choose files to upload (a zip of finetune/, or the 3 loose files)...")
uploaded = files.upload()

# Stage everything (zip contents + any loose files) into one temp tree.
stage = Path("_staged")
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()
for fn in list(uploaded):
    if fn.lower().endswith(".zip"):
        with zipfile.ZipFile(fn) as z:
            z.extractall(stage)
        os.remove(fn)
    elif os.path.exists(fn):
        shutil.move(fn, stage / os.path.basename(fn))

# Find each required file ANYWHERE in the staged tree (robust to zip layout) and copy it.
targets = {
    "colab_train.py":  "finetune/colab_train.py",
    "sft_train.jsonl": "finetune/dataset/sft_train.jsonl",
    "sft_eval.jsonl":  "finetune/dataset/sft_eval.jsonl",
}
for name, dest in targets.items():
    hit = next(stage.rglob(name), None)
    if hit:
        shutil.copy(hit, dest)

missing = [d for d in targets.values() if not os.path.exists(d)]
if missing:
    print("Could not place:", missing)
    print("Files seen inside your upload:", [p.name for p in stage.rglob('*') if p.is_file()])
shutil.rmtree(stage, ignore_errors=True)
assert not missing, f"Still missing: {missing}. Check the file list above, then re-run."
print("\nReady:")
!wc -l finetune/dataset/sft_train.jsonl finetune/dataset/sft_eval.jsonl

In [ ]:
#@title 3. Install Unsloth + training stack (~2-3 min)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 137.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.7 MB/s eta 0:00:00


In [ ]:
#@title 4. Train + export GGUF (T4 defaults: max_seq=2048, bsz 2x4, 3 epochs)
# Building llama.cpp for the GGUF export adds a few minutes the first time.
!python finetune/colab_train.py --data finetune/dataset/sft_train.jsonl

python3: can't open file '/content/finetune/colab_train.py': [Errno 2] No such file or directory


In [ ]:
#@title 5. Download the Q4_K_M GGUF (save it to finetune/export/qwen-forgesight/ locally)
import glob
from google.colab import files

gguf = glob.glob("finetune/export/**/*.Q4_K_M.gguf", recursive=True) \
    or glob.glob("finetune/export/**/*.gguf", recursive=True)
assert gguf, "No GGUF found — check the training cell output for export errors."
print("Downloading:", gguf[0], f"({os.path.getsize(gguf[0]) / 1e9:.2f} GB)")
files.download(gguf[0])

AssertionError: No GGUF found — check the training cell output for export errors.

## Back on your laptop (RTX 3060 + Ollama)

```bash
# 1. Put the downloaded file here (the Modelfile points at this exact path):
#    finetune/export/qwen-forgesight/unsloth.Q4_K_M.gguf
ollama create qwen-forgesight -f finetune/Modelfile

# 2. Measure base vs fine-tune (citation compliance + number fidelity):
python finetune/03_evaluate_vs_base.py --model qwen2.5:3b-instruct
python finetune/03_evaluate_vs_base.py --model qwen-forgesight

# 3. Promote ONLY if the fine-tune wins: set OLLAMA_MODEL=qwen-forgesight and
#    SYNTHESIS_BACKEND=ollama in your local .env. (Railway stays on Groq — no GPU in cloud.)
```

Record the eval JSON in `docs/finetune.md`. The public Railway URL keeps using Groq; the
fine-tuned SLM is the on-prem local path.